In [1]:
import os
import torch
#from diffusers import StableDiffusion3Pipeline
from diffusers import StableDiffusionPipeline
from huggingface_hub import login
import json 



with open("/workspace/data/huggingface/__token__.json", "r") as file: 
    data = json.load(file)

login(data['token'])

print(torch.cuda.is_available())
print(torch.cuda.device_count())

pipe = StableDiffusionPipeline.from_pretrained("hf-internal-testing/tiny-stable-diffusion-torch", torch_dtype=torch.float32, use_safetensors=False )



/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True
1


Loading weights: 100%|██████████| 84/84 [00:00<00:00, 23374.35it/s]
CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--hf-internal-testing--tiny-stable-diffusion-torch/snapshots/a88cdfbd91f96ec7f61eb7484b652ff0f4ee701d/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 20.12it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend 

In [7]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

# 1. Load Model & Processor
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base',
    torch_dtype=torch.float16, 
    use_safetensors=True
).to('cuda') # Ensure model is moved to GPU

# 2. Load Image
img = Image.open('/workspace/data/wild/hienna.png').convert('RGB')

# 3. Process - FIXED: return_tensors (plural)
inputs = processor(images=img, return_tensors='pt').to('cuda', torch.float16)

# 4. Generate
# Using torch.no_grad() saves extra VRAM on your 1050
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=50)

# 5. Result
caption = processor.decode(out[0], skip_special_tokens=True)
print(f"Generated Caption: {caption}")

Loading weights: 100%|██████████| 472/472 [00:00<00:00, 3580.19it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated Caption: a hyene eating a dead animal


In [8]:
out

tensor([[30522,  1037,  1044, 20684,  2063,  5983,  1037,  2757,  4111,   102]],
       device='cuda:0')

In [9]:
inputs


{'pixel_values': tensor([[[[ 1.9160,  1.9307,  1.9160,  ...,  1.9014,  1.9014,  1.8721],
          [ 1.9307,  1.9307,  1.9307,  ...,  1.9014,  1.9014,  1.8867],
          [ 1.9307,  1.9307,  1.9307,  ...,  1.9014,  1.8867,  1.9014],
          ...,
          [-0.2739, -1.1357, -0.7119,  ...,  0.3975,  0.0179,  0.2369],
          [-0.2739, -0.6826, -0.2886,  ...,  0.3684,  0.2078,  0.5435],
          [ 0.0179, -0.1572, -0.0258,  ...,  0.5142,  0.4121,  0.6309]],

         [[ 1.7451,  1.7451,  1.7598,  ...,  2.0156,  2.0293,  2.0293],
          [ 1.7295,  1.7295,  1.7451,  ...,  2.0293,  2.0293,  2.0293],
          [ 1.7295,  1.7451,  1.7451,  ...,  2.0449,  2.0293,  2.0293],
          ...,
          [-0.5513, -1.3018, -0.9268,  ..., -0.0412, -0.3713, -0.1013],
          [-0.5513, -0.8818, -0.5215,  ..., -0.0562, -0.1913,  0.2139],
          [-0.2512, -0.3564, -0.2664,  ...,  0.0789,  0.0638,  0.3340]],

         [[ 1.3496,  1.3779,  1.3779,  ...,  2.0742,  2.0742,  2.0312],
          [ 1